# Phase 2: Professional Feature Engineering Pipeline (Expert Edition)

This notebook implements an advanced, leakage-free pipeline for climate-health impact modelling.
**Key Principles:**

1. **Temporal Continuity:** Computing lags and rolling features on a global continuous dataframe before splitting.
2. **Strict Anti-Leakage:** Deriving medians and scaling parameters solely from the pre-2023 training period.
3. **Data Integrity:** Ensuring no test data is lost due to edge-case NaNs at the split boundary.


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import os
import joblib

# Configuration
DATA_PATH = "../data/raw/global_climate_health_impact_tracker_2015_2025.csv"
np.random.seed(42)
pd.set_option('display.max_columns', None)

# Load and initial sort for global continuity
df_raw = pd.read_csv(DATA_PATH, parse_dates=['date'])
df_raw = df_raw.sort_values(['country_name', 'date']).reset_index(drop=True)
print(f"Global dataset loaded: {df_raw.shape}")

Global dataset loaded: (14100, 30)


## 1. Data Integrity Strategy (Anti-Leakage)

Calculating medians from the training period ONLY, then applying global integrity fixes to the entire dataset to maintain continuity.


In [2]:
# Step 1: Calculate Medians from Train period (< 2023) ONLY
train_period_mask = df_raw['date'] < '2023-01-01'
df_train_ref = df_raw[train_period_mask]

# Spatio-temporal group medians
medians = df_train_ref.groupby(['region', df_raw['date'].dt.month])['air_quality_index'].median().reset_index()
medians.rename(columns={'date': 'month'}, inplace=True)
# print(medians)
# Global fallback median (Expert addition for robustness)
global_median_fallback = df_train_ref['air_quality_index'].median()

# Step 2: Global integrity application
def fix_global_integrity(df, medians_ref, fallback):
    df['month'] = df['date'].dt.month
    df = df.merge(medians_ref, on=['region', 'month'], how='left', suffixes=('', '_median'))
    
    # Impute anomalies using prioritized hierarchy: Group Median -> Global Fallback
    mask_anomaly = (df['air_quality_index'] < 0) | (df['air_quality_index'].isna())
    df.loc[mask_anomaly, 'air_quality_index'] = df['air_quality_index_median'].fillna(fallback)
    
    df = df.drop(columns=['air_quality_index_median'])
    
    # Domain-specific range capping
    health_cols = ['vector_disease_risk_score', 'respiratory_disease_rate', 'waterborne_disease_incidents']
    df[health_cols] = df[health_cols].clip(0, 100)
    
    return df.sort_values(['country_name', 'date']).reset_index(drop=True)

df_cleaned = fix_global_integrity(df_raw, medians, global_median_fallback)
print(f"Global integrity mapped with fallback ({global_median_fallback}). Shape: {df_cleaned.shape}")

Global integrity mapped with fallback (93.0). Shape: (14100, 30)


## 2. Global Feature Engineering

Computing lags and rolling windows on the full continuous dataframe to eliminate edge-case NaNs at the split boundary.


In [3]:
# 2.1 Temporal Lag Features (Climate + Health Indicators)
# We lag health indicators to prevent look-ahead bias in forecasting
for lag in [1, 2]:
    # Climate lags
    for col in ['temperature_celsius', 'precipitation_mm', 'pm25_ugm3']:
        df_cleaned[f'{col}_lag_{lag}'] = df_cleaned.groupby('country_name')[col].shift(lag)
    
    # Health lags (Critical for anti-leakage in forecasting)
    for col in ['respiratory_disease_rate', 'waterborne_disease_incidents']:
        df_cleaned[f'{col}_lag_{lag}'] = df_cleaned.groupby('country_name')[col].shift(lag)

# 2.2 Rolling Window Features (4-week Accumulation)
# A. Lượng mưa (Precipitation): Mean, Sum (Tích lũy nước) & Std (Biến động lượng mưa)
for agg in ['mean', 'sum']:
    df_cleaned[f'precip_rolling_4wk_{agg}'] = df_cleaned.groupby('country_name')['precipitation_mm'].transform(lambda x: x.rolling(4).agg(agg))
df_cleaned['precip_rolling_4wk_std'] = df_cleaned.groupby('country_name')['precipitation_mm'].transform(lambda x: x.rolling(4).std())

# B. Nhiệt độ (Temperature): Mean (Nền nhiệt trung bình) & Std (Độ biến động thời tiết/Sốc nhiệt)
df_cleaned['temp_rolling_4wk_mean'] = df_cleaned.groupby('country_name')['temperature_celsius'].transform(lambda x: x.rolling(4).mean())
df_cleaned['temp_rolling_4wk_std'] = df_cleaned.groupby('country_name')['temperature_celsius'].transform(lambda x: x.rolling(4).std())

# C. Chất lượng không khí (PM2.5): Mean (Mức độ phơi nhiễm lâu dài) & Std (Mức độ trồi sụt ô nhiễm)
df_cleaned['pm25_rolling_4wk_mean'] = df_cleaned.groupby('country_name')['pm25_ugm3'].transform(lambda x: x.rolling(4).mean())
df_cleaned['pm25_rolling_4wk_std'] = df_cleaned.groupby('country_name')['pm25_ugm3'].transform(lambda x: x.rolling(4).std())

# 2.3 Cyclical Encoding
df_cleaned['month_sin'] = np.sin(2 * np.pi * df_cleaned['month'] / 12)
df_cleaned['month_cos'] = np.cos(2 * np.pi * df_cleaned['month'] / 12)

# 2.4 Interaction Features
income_map = {'High': 3, 'Upper-Middle': 2, 'Lower-Middle': 1, 'Low': 0}
df_cleaned['income_encoded'] = df_cleaned['income_level'].map(income_map)
df_cleaned['precip_vulnerability_interact'] = df_cleaned['precip_rolling_4wk_mean'] * (3 - df_cleaned['income_encoded'])

# 2.5 Redundancy & Leakage Reduction
# Drop original non-lagged auxiliary health features to force model to use temporal patterns
cols_to_drop = ['air_quality_index', 'respiratory_disease_rate', 'waterborne_disease_incidents']
df_cleaned = df_cleaned.drop(columns=cols_to_drop)

print(f"Global feature engineering complete. Columns: {len(df_cleaned.columns)}")

Global feature engineering complete. Columns: 48


In [4]:
df_cleaned.columns

Index(['record_id', 'country_code', 'country_name', 'region', 'income_level',
       'date', 'year', 'month', 'week', 'latitude', 'longitude',
       'population_millions', 'temperature_celsius', 'temp_anomaly_celsius',
       'precipitation_mm', 'heat_wave_days', 'drought_indicator',
       'flood_indicator', 'extreme_weather_events', 'pm25_ugm3',
       'cardio_mortality_rate', 'vector_disease_risk_score',
       'heat_related_admissions', 'healthcare_access_index',
       'gdp_per_capita_usd', 'mental_health_index', 'food_security_index',
       'temperature_celsius_lag_1', 'precipitation_mm_lag_1',
       'pm25_ugm3_lag_1', 'respiratory_disease_rate_lag_1',
       'waterborne_disease_incidents_lag_1', 'temperature_celsius_lag_2',
       'precipitation_mm_lag_2', 'pm25_ugm3_lag_2',
       'respiratory_disease_rate_lag_2', 'waterborne_disease_incidents_lag_2',
       'precip_rolling_4wk_mean', 'precip_rolling_4wk_sum',
       'precip_rolling_4wk_std', 'temp_rolling_4wk_mean',
       

## 3. Boundary Preservation & Final Export

Splitting at 2023, applying dropna to the global pool, and verifying test set integrity (No 2023 rows lost). Training-based Robust Scaling is then applied to prevent information leakage.


In [ ]:
from sklearn.preprocessing import PowerTransformer # Đừng quên import ở đầu file
# pt = PowerTransformer(method='yeo-johnson')
# 3.1 Drop NaNs globally (Removes pre-2015/missing starts)
df_final = df_cleaned.dropna()

# 3.1b Memory Optimization (Expert standard)
numeric_cols = df_final.select_dtypes(include=[np.number]).columns
df_final[numeric_cols] = df_final[numeric_cols].astype('float32')

# 3.2 Temporal Split
df_train = df_final[df_final['date'] < '2023-01-01'].copy()
df_test = df_final[df_final['date'] >= '2023-01-01'].copy()

# 3.3 Stateful Robust Scaling
target_cols = ['vector_disease_risk_score']
cyclical_cols = ['month_sin', 'month_cos']
categorical_encoded = ['income_encoded', 'month']

features_to_scale = [
    col for col in df_train.select_dtypes(include=[np.number]).columns 
    if col not in (target_cols + cyclical_cols + categorical_encoded)
]

scaler = RobustScaler()
df_train[features_to_scale] = scaler.fit_transform(df_train[features_to_scale])
df_test[features_to_scale] = scaler.transform(df_test[features_to_scale])

# 3.4 Target Distribution Alignment
df_train['target_log'] = np.log1p(df_train['vector_disease_risk_score'])
df_test['target_log'] = np.log1p(df_test['vector_disease_risk_score'])

# df_train['target_log'] = pt.fit_transform(df_train[['vector_disease_risk_score']])
# df_test['target_log'] = pt.transform(df_test[['vector_disease_risk_score']])

# 3.5 Persistency & Export
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../artifacts', exist_ok=True)

# Export Data
df_train.to_csv('../data/processed/train_features.csv', index=False)
df_test.to_csv('../data/processed/test_features.csv', index=False)

# Export Scaler (For future inference)
joblib.dump(scaler, '../artifacts/robust_scaler.joblib')
# joblib.dump(pt, '../artifacts/target_scaler.joblib')

print(f"Data Continuity Verification:")
print(f"- Training Samples: {len(df_train)}")
print(f"- Test Samples (2023+): {len(df_test)}")
print(f"- Memory Optimized: float32")
print(f"- Scaler exported to ../artifacts/robust_scaler.joblib")
print("Pipeline complete. Ready for modeling.")

Data Continuity Verification:
- Training Samples: 10350
- Test Samples (2023+): 3675
- Memory Optimized: float32
- Scaler exported to ../models/robust_scaler.joblib
Pipeline complete. Ready for modeling.
